# 第0阶段之v2

## 8. 修正：重建V2 并冻结

机器学习项目，不能做完实验凭感觉写结论，要把所有规则写成一张"标准说明书（口径规格表）"，交给 AI / 代码去自动执行、自动检查，保证以后任何人复现、迭代都不走样。

很多同学做毕设、做机器学习项目，踩一个大坑：代码写得乱糟糟，清洗、筛选、打标签的规则零散藏在代码里面，自己当时知道怎么选样本，过两周就忘了；换一个人跑代码，稍微改一行，样本就变了，模型结果直接变，谁也说不清差异是模型导致的，还是数据筛选规则偷偷变了。这张表，就是把咱们前面所有复盘、踩过的坑，全部固化成白纸黑字的硬性规则，也就是数据契约。

### 例如：通过AI实现上面的规范

按下面的规格表，从 incident.db 重新构建训练集和验证集。这是一个新的 Notebook，不是修改已有代码——按表重写。

【环境】Python 3.12，pandas 3.x；数据库 ../data/incident.db；产物写到 ../output/v2/（不存在就建）。

三张表：
- residents(resident_id, gender, birth_date, education, income_level, district, enroll_date)
- visits(visit_id, resident_id, visit_date, created_at, known_diabetes, chronic_mgmt_tag, height_cm, weight_kg, waist_cm, sbp, dbp, smoking_status, drinking_freq, exercise_min_week, sleep_hours, veg_fruit_daily, family_history_dm, hypertension_dx, hyperlipidemia_dx, gdm_history, polyuria, polydipsia, weight_loss_unexplained)
- lab_results(lab_id, resident_id, sample_date, report_date, fpg_mmol_l, hba1c_pct)

已知的写法问题：gender 里有 male、female、男、女；smoking_status 有大小写不一、首尾空格，以及 从不、吸烟、已戒；drinking_freq 有 不喝、偶尔、经常。

【规格表】
- 筛查人群：只保留 known_diabetes == 0 的随访；该列不进输出
- 标签：随访日起 90 天内（含当天）的化验，取采样最早的一条（同日再按 lab_id）；空腹血糖 >= 7.0 或糖化 >= 6.5 为 1，否则为 0
- 没化验：窗口内没有已出报告的化验 → 该随访不进数据集（不是阴性）
- 训练快照日：2024-06-30，训练集只能用这一天已经出报告的化验
- 训练随访最晚：2024-06-30 −（90 + 30）天 = 2024-03-02
- 验证区间：2024-07-01 ～ 2024-12-31（标签用导出日 2025-06-30 已出的报告）
- 隔离期：2024-03-03 ～ 2024-06-30 的随访两边都不要
- 2025 年数据：不用
- 重复提交：同一居民同一天只保留 created_at 最晚的一条（排序键 created_at、visit_id）
- 单位修正：身高 < 3 → ×100；体重与该居民此前随访体重中位数之比在 1.8～2.2 → ÷2；首次随访（此前没有体重）用 BMI > 40 判断，BMI 用修正后的身高算
- 哨兵与越界：收缩压 0 或 999 → 空；数值超出下面的范围 → 空（不删行）
    height_cm 120–210，weight_kg 30–200，sbp 60–260，dbp 30–150，exercise_min_week 0–3000，sleep_hours 2–14，waist_cm 50–150
- 类别写法：gender 只能是 M/F；smoking_status 只能是 never/former/current；drinking_freq 只能是 never/occasional/frequent；family_history_dm 只能是 yes/no/unknown（unknown 保留）
- 缺失：保留缺失，不填补、不删行（填补是模型的事）
- 年龄：随访日的周岁，整数：(visit_date − birth_date) 的天数 // 365.25，再转 int；应在 18–100 岁之间
- 输出列：19 个输入 + label + 6 个元数据 = 26 列（列名和顺序见要求 4）
- 输出文件：train.parquet、valid.parquet、summary.json
- 自检：visit_id 各自唯一且不跨集；训练最晚日期 <= 2024-03-02；验证全部在区间内；label 只有 0/1

【要求】
1. 严格按表实现。表里没写到的地方不要自作主张；必须做决定时，在代码注释里写"规格未定义，我选择了……"，并在最后单独列出所有这类决定。
2. 分成多个代码格，每格只做一件事，按这个顺序：导入 → 口径常量 → 读库与写法统一 → 去重 → 单位、哨兵、越界 → 标签函数 → 人群与时点切分 → 拼档案、算年龄、只留 26 列 → 落盘前自检 → 落盘与 summary.json。去重和单位修正在全部随访上做（包括已确诊者），之后才筛人群。
3. 所有口径数字（窗口天数、报告缓冲、快照日、验证区间、阳性阈值、取值范围、年龄范围）定义成常量，集中在第二格；代码里不出现裸数字。训练随访最晚日期由常量算出来，不要手写。
4. 列用显式白名单。输出列的名字和顺序固定为：
   age, height_cm, weight_kg, waist_cm, sbp, dbp, exercise_min_week, sleep_hours, veg_fruit_daily, hypertension_dx, hyperlipidemia_dx, gdm_history, polyuria, polydipsia, weight_loss_unexplained, gender, smoking_status, drinking_freq, family_history_dm, label, visit_id, resident_id, visit_date, district, education, income_level
   known_diabetes、chronic_mgmt_tag 不得出现在输出里。两份表都按 visit_id 排序。
5. 写法统一之后，用 assert 检查每个类别列的取值都在规格表允许的集合里；漏了一种写法要报错停下。
6. 每一步清洗都打印"改了多少条"，0 条也要打印。
7. 缺失值保留，不填补、不删行；越界值置空，不删行。
8. 落盘前用 assert 检查：visit_id 各自唯一且两集无交集；训练最晚日期 <= 训练随访最晚日；验证全部在验证区间内；label 只有 0/1。
9. 输出 train.parquet、valid.parquet、summary.json，并把 summary.json 打印出来。
   summary.json 的结构固定为：
   {"source_db", "duplicates_removed", "weight_unit_fixed", "train_last_visit_allowed", "train": {"rows", "positives", "positive_rate", "date_min", "date_max", "residents"}, "valid": {同上}, "train_valid_resident_overlap"}
10. 不训练模型，不做独热编码，不画图。


## 8.1 创建新的文件：第0阶段之v2.ipynb

### 导入工具包

In [1]:
import pandas as pd
import numpy as np
import sqlite3                                              # 读 SQLite 数据库
import json                                                 # 写 summary.json、FROZEN.json
import os                                                   # 建目录、列文件
import shutil                                               # 复制文件（冻结时用）
import hashlib                                              # 算指纹（冻结时用）
import platform                                             # 记下 Python 版本
from sklearn.pipeline import Pipeline                       # 搭建流水线
from sklearn.compose import ColumnTransformer               # 三组列分开处理
from sklearn.impute import SimpleImputer                    # 填补缺失（在模型里做，不在数据集里做）
from sklearn.preprocessing import OneHotEncoder             # 对数据做独热编码
from sklearn.preprocessing import StandardScaler            # 对数据做标准化
from sklearn.linear_model import LogisticRegression         # 逻辑回归
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score   # 准确率、AUC、AP

### 口径常量：复盘的结论全部写在这里

In [2]:
DATA = "../data/"
DB = DATA + "incident.db"
OUT = "../output/v2/"                                       # 产物不再写到 Notebook 旁边
FROZEN = DATA + "frozen/"                                   # 冻结集：交给第一轮骨架的那一份（8.7–8.10）
TARGET = "label"

WINDOW_DAYS = 90                                            # 随访后多少天内的化验算数
MAX_REPORT_LAG_DAYS = 30                                    # 报告最晚比采样晚多少天（标签成熟缓冲）
TRAIN_END = pd.Timestamp("2024-06-30")                      # 训练快照日：只用这天已经知道的信息
VALID_START, VALID_END = pd.Timestamp("2024-07-01"), pd.Timestamp("2024-12-31")
DATA_AS_OF = pd.Timestamp("2025-06-30")                     # 原始库的导出日
FPG_POS, HBA1C_POS = 7.0, 6.5                               # 阳性阈值
TRAIN_LAST = TRAIN_END - pd.Timedelta(days=WINDOW_DAYS + MAX_REPORT_LAG_DAYS)   # 训练随访最晚的一天

NUM_COLS = ["age", "height_cm", "weight_kg", "waist_cm", "sbp", "dbp", "exercise_min_week", "sleep_hours"]
BIN_COLS = ["veg_fruit_daily", "hypertension_dx", "hyperlipidemia_dx", "gdm_history",
            "polyuria", "polydipsia", "weight_loss_unexplained"]
CAT_COLS = ["gender", "smoking_status", "drinking_freq", "family_history_dm"]
META_COLS = ["visit_id", "resident_id", "visit_date", "district", "education", "income_level"]
COLUMNS = NUM_COLS + BIN_COLS + CAT_COLS + [TARGET] + META_COLS                 # 26 列
ENUMS = {"gender": ["M", "F"], "smoking_status": ["never", "former", "current"],
         "drinking_freq": ["never", "occasional", "frequent"], "family_history_dm": ["yes", "no", "unknown"]}
RANGES = {"height_cm": (120, 210), "weight_kg": (30, 200), "sbp": (60, 260), "dbp": (30, 150),
          "exercise_min_week": (0, 3000), "sleep_hours": (2, 14), "waist_cm": (50, 150)}
AGE_RANGE = (18, 100)                                       # 筛查对象的年龄范围（成年人）
print("训练随访最晚:", TRAIN_LAST.date(), "  输出列数:", len(COLUMNS))

训练随访最晚: 2024-03-02   输出列数: 26


### 读库 + 类别写法统一

In [3]:
with sqlite3.connect(DB) as conn:
    residents = pd.read_sql("SELECT * FROM residents", conn, parse_dates=["birth_date"])
    visits = pd.read_sql("SELECT * FROM visits", conn, parse_dates=["visit_date", "created_at"])
    labs = pd.read_sql("SELECT * FROM lab_results", conn, parse_dates=["sample_date", "report_date"])

residents["gender"] = residents["gender"].replace({"male": "M", "男": "M", "female": "F", "女": "F"})
visits["smoking_status"] = (visits["smoking_status"].str.strip().str.lower()
                            .replace({"从不": "never", "吸烟": "current", "已戒": "former"}))
visits["drinking_freq"] = visits["drinking_freq"].replace({"不喝": "never", "偶尔": "occasional", "经常": "frequent"})

assert residents["gender"].isin(ENUMS["gender"]).all()                            # 写法统一了没有，让代码说
assert visits["smoking_status"].isin(ENUMS["smoking_status"]).all()
assert visits["drinking_freq"].isin(ENUMS["drinking_freq"]).all()
assert visits["family_history_dm"].isin(ENUMS["family_history_dm"]).all()
print(visits.shape)

(101297, 23)


### 重复提交：同一人同一天只留最后录入的一条

In [4]:
n0 = len(visits)
visits["_day"] = visits["visit_date"].dt.normalize()
visits = (visits.sort_values(["created_at", "visit_id"])                     # 排序键写全，谁是"最后一条"才唯一
          .drop_duplicates(["resident_id", "_day"], keep="last")
          .drop(columns="_day"))
n_dup = n0 - len(visits)
print("去掉重复提交:", n_dup, "条")

去掉重复提交: 437 条


### 单位、哨兵、越界：只用"当天及以前"的信息

In [5]:
visits = visits.sort_values(["resident_id", "visit_date"]).reset_index(drop=True)

n_height = (visits["height_cm"] < 3).sum()
visits.loc[visits["height_cm"] < 3, "height_cm"] *= 100                       # 按米录入
prior = (visits.groupby("resident_id")["weight_kg"]
         .transform(lambda s: s.shift().expanding().median()))                # 此前随访的体重中位数
bmi = visits["weight_kg"] / (visits["height_cm"] / 100) ** 2
jin = (visits["weight_kg"] / prior).between(1.8, 2.2) | (prior.isna() & (bmi > 40))  # 首次用 BMI 兜底
visits.loc[jin, "weight_kg"] /= 2                                             # 按斤录入

n_sentinel = visits["sbp"].isin([0, 999]).sum()
visits.loc[visits["sbp"].isin([0, 999]), "sbp"] = np.nan                      # 哨兵值
n_range = {}
for col, (lo, hi) in RANGES.items():                                          # 越界一律置空，不删行
    bad = visits[col].notna() & ~visits[col].between(lo, hi)
    n_range[col] = int(bad.sum())
    visits.loc[bad, col] = np.nan
print({"身高×100": int(n_height), "体重÷2": int(jin.sum()), "哨兵置空": int(n_sentinel)})
print("越界置空:", n_range)

{'身高×100': 610, '体重÷2': 520, '哨兵置空': 278}
越界置空: {'height_cm': 0, 'weight_kg': 0, 'sbp': 0, 'dbp': 0, 'exercise_min_week': 218, 'sleep_hours': 0, 'waist_cm': 31}


### 标签：截至 as_of 已出报告、随访后 90 天内的首次化验

In [6]:
def attach_label(v, as_of):
    """给随访挂标签。窗口内没有已出报告的化验 → 这条随访不返回（没化验 ≠ 阴性）。"""
    known = labs[labs["report_date"] <= as_of]                      # 站在 as_of 那天能看到的化验
    m = v[["visit_id", "resident_id", "visit_date"]].merge(known, on="resident_id")
    m = m[(m["sample_date"] >= m["visit_date"])
          & (m["sample_date"] <= m["visit_date"] + pd.Timedelta(days=WINDOW_DAYS))]
    first = m.sort_values(["sample_date", "lab_id"]).drop_duplicates("visit_id")  # 排序键写全，结果才唯一
    first[TARGET] = ((first["fpg_mmol_l"] >= FPG_POS) | (first["hba1c_pct"] >= HBA1C_POS)).astype(int)
    return v.merge(first[["visit_id", TARGET]], on="visit_id", how="inner")

### 人群 + 时点切分

In [7]:
screen = visits[visits["known_diabetes"] == 0]                                   # 已确诊者不是筛查对象
train = attach_label(screen[screen["visit_date"] <= TRAIN_LAST], as_of=TRAIN_END)
valid = attach_label(screen[screen["visit_date"].between(VALID_START, VALID_END)], as_of=DATA_AS_OF)
print("训练", len(train), "  验证", len(valid))

训练 32632   验证 13067


### 拼档案、算年龄、只留 26 列

In [8]:
def finalize(d):
    d = d.merge(residents, on="resident_id", how="left")
    d["age"] = ((d["visit_date"] - d["birth_date"]).dt.days // 365.25).astype(int)   # 随访日的周岁
    return d[COLUMNS].sort_values("visit_id").reset_index(drop=True)

train, valid = finalize(train), finalize(valid)
print(train.shape, valid.shape)

(32632, 26) (13067, 26)


### 落盘前自检：写错了就在这里炸

In [9]:
assert train["visit_id"].is_unique and valid["visit_id"].is_unique
assert not set(train["visit_id"]) & set(valid["visit_id"]), "同一次随访跨集"
assert train["visit_date"].max() <= TRAIN_LAST, "训练集含标签未成熟的随访"
assert valid["visit_date"].between(VALID_START, VALID_END).all(), "验证集超出区间"
assert train[TARGET].isin([0, 1]).all() and valid[TARGET].isin([0, 1]).all()
assert "chronic_mgmt_tag" not in COLUMNS and "known_diabetes" not in COLUMNS
print("五条自检全部通过")

五条自检全部通过


### 落盘 + 数据说明 summary.json

In [10]:
os.makedirs(OUT, exist_ok=True)
train.to_parquet(OUT + "train.parquet", index=False)
valid.to_parquet(OUT + "valid.parquet", index=False)

def describe(d):
    return {"rows": len(d), "positives": int(d[TARGET].sum()),
            "positive_rate": round(float(d[TARGET].mean()), 4),
            "date_min": str(d["visit_date"].min().date()), "date_max": str(d["visit_date"].max().date()),
            "residents": int(d["resident_id"].nunique())}
summary = {"source_db": "incident.db", "duplicates_removed": n_dup, "weight_unit_fixed": int(jin.sum()),
           "train_last_visit_allowed": str(TRAIN_LAST.date()),
           "train": describe(train), "valid": describe(valid),
           "train_valid_resident_overlap": len(set(train["resident_id"]) & set(valid["resident_id"]))}
json.dump(summary, open(OUT + "summary.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "source_db": "incident.db",
  "duplicates_removed": 437,
  "weight_unit_fixed": 520,
  "train_last_visit_allowed": "2024-03-02",
  "train": {
    "rows": 32632,
    "positives": 5285,
    "positive_rate": 0.162,
    "date_min": "2023-01-01",
    "date_max": "2024-03-02",
    "residents": 26565
  },
  "valid": {
    "rows": 13067,
    "positives": 1675,
    "positive_rate": 0.1282,
    "date_min": "2024-07-01",
    "date_max": "2024-12-31",
    "residents": 12806
  },
  "train_valid_resident_overlap": 10633
}


## 8.2 验收清单

### 在 v2 上训练同一个模型

In [11]:
X_train, y_train = train[NUM_COLS + BIN_COLS + CAT_COLS], train[TARGET]
X_valid, y_valid = valid[NUM_COLS + BIN_COLS + CAT_COLS], valid[TARGET]
encoder = OneHotEncoder(handle_unknown="ignore")
pipe = Pipeline([
    ("pre", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),  # 缺失也是信息
                          ("scale", StandardScaler())]), NUM_COLS),
        ("bin", SimpleImputer(strategy="constant", fill_value=0, add_indicator=True), BIN_COLS),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                          ("onehot", encoder)]), CAT_COLS),
    ])),
    ("lr", LogisticRegression(max_iter=2000)),
])
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_valid)[:, 1]
k = int(round(len(valid) * 0.15))                                 # 15% 名额
hit = y_valid.to_numpy()[np.argsort(-proba)[:k]].sum()
print({"患病率": round(float(y_valid.mean()), 3),
       "准确率@0.5": round(accuracy_score(y_valid, proba >= 0.5), 3),
       "ROC-AUC": round(roc_auc_score(y_valid, proba), 4),
       "AP": round(average_precision_score(y_valid, proba), 4),
       "R@15": round(float(hit / y_valid.sum()), 3), "P@15": round(float(hit / k), 3)})

{'患病率': 0.128, '准确率@0.5': 0.929, 'ROC-AUC': 0.9329, 'AP': 0.7665, 'R@15': 0.747, 'P@15': 0.639}


## 8.3 冻结

验收清单回答的是"口径对不对"，所以允许 ±1% 的出入。接下来要回答的是另一个问题：后面所有的工作，用的是不是同一份数据。第一轮要照着文档一步步复现，文档里写"训练 32,632 条、阳性 5,285"，每个人的 manifest 都得是这个数；第二轮要把 v2 搬进框架、从 incident.db 再重建一遍，重建的结果要和今天这份逐格比对。两件事都需要一份定下来、不再改的数据。把它定下来，叫冻结。

### 冻结前：按下游读数据时会查的，再查一遍

In [12]:
def check_frozen(d):
    """冻结前的入口检查：列、编号、标签、写法、取值范围。有问题一次列全，不在第一个问题就停。"""
    if list(d.columns) != COLUMNS:
        return [f"列不对：多了 {sorted(set(d.columns) - set(COLUMNS))}，少了 {sorted(set(COLUMNS) - set(d.columns))}"]
    problems = []
    if d["visit_id"].isna().any() or not d["visit_id"].is_unique:
        problems.append("visit_id 有缺失或重复")
    if not d[TARGET].isin([0, 1]).all():
        problems.append("label 不全是 0/1")
    if not pd.api.types.is_datetime64_any_dtype(d["visit_date"]):
        problems.append("visit_date 不是日期类型")
    for col, allowed in ENUMS.items():                                   # 类别：只能是规定的写法，不能缺
        n = int((~d[col].isin(allowed)).sum())
        if n:
            problems.append(f"{col}: {n} 行不在 {allowed} 里")
    for col in BIN_COLS:                                                 # 二元：只能是 0/1，允许缺
        n = int((d[col].notna() & ~d[col].isin([0, 1])).sum())
        if n:
            problems.append(f"{col}: {n} 行不是 0/1")
    for col, (lo, hi) in {**RANGES, "age": AGE_RANGE}.items():           # 数值：在范围内，允许缺
        n = int((d[col].notna() & ~d[col].between(lo, hi)).sum())
        if n:
            problems.append(f"{col}: {n} 行超出 [{lo}, {hi}]")
    return problems

train = pd.read_parquet(OUT + "train.parquet")                           # 查的是验收过的那两份文件
valid = pd.read_parquet(OUT + "valid.parquet")
print("train 的问题:", check_frozen(train))
print("valid 的问题:", check_frozen(valid))
missing = pd.DataFrame({"train": train.isna().mean(), "valid": valid.isna().mean()}).round(3)
print(missing[(missing["train"] > 0) | (missing["valid"] > 0)])          # 只看有缺失的列

train 的问题: []
valid 的问题: []
                         train  valid
waist_cm                 0.152  0.151
sbp                      0.003  0.003
exercise_min_week        0.002  0.002
sleep_hours              0.080  0.081
gdm_history              0.480  0.478
polyuria                 0.030  0.027
polydipsia               0.029  0.030
weight_loss_unexplained  0.030  0.031
income_level             0.171  0.168


### 两种指纹：内容指纹、文件指纹

In [13]:
def content_sha256(d):
    """内容指纹：固定列序、数值统一成浮点、按 visit_id 排好，写成 CSV 文本再算 sha256。
    只认内容：文件格式、库版本、行的顺序、存成整数还是浮点，都不影响它。"""
    canon = d[COLUMNS].astype({c: "float64" for c in NUM_COLS + BIN_COLS + [TARGET]})
    text = canon.sort_values("visit_id").to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def file_sha256(path):
    """文件指纹：逐块读文件算 sha256。用来记住原始库是哪一份。"""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

print("train 内容指纹:", content_sha256(train)[:16])
print("valid 内容指纹:", content_sha256(valid)[:16])
print("incident.db 文件指纹:", file_sha256(DB)[:16])
print("打乱行序后:", content_sha256(train.sample(frac=1, random_state=0))[:16])     # 行序不影响内容指纹

train 内容指纹: 52e41f28ec7da2b0
valid 内容指纹: b489f7c685ee85ec
incident.db 文件指纹: 412b7c1adb612f9f
打乱行序后: 52e41f28ec7da2b0


### 写出冻结集：两份数据 + 一张身份证 FROZEN.json

着重看 身份信息文件：FROZEN.json，这里面就是 口径。

In [14]:
card = {
    "name": "stage0-v2",
    "built_by": "第 0 阶段 v2 Notebook 单元 38–47（口径见规格表 8.1）",
    "source": {"file": "incident.db", "sha256": file_sha256(DB)},
    "spec": {"population": "known_diabetes == 0", "window_days": WINDOW_DAYS,
             "max_report_lag_days": MAX_REPORT_LAG_DAYS, "fpg_pos": FPG_POS, "hba1c_pos": HBA1C_POS,
             "train_end": str(TRAIN_END.date()), "train_last_visit": str(TRAIN_LAST.date()),
             "valid_start": str(VALID_START.date()), "valid_end": str(VALID_END.date()),
             "data_as_of": str(DATA_AS_OF.date())},
    "columns": {"numeric": NUM_COLS, "binary": BIN_COLS, "categorical": CAT_COLS,
                "label": TARGET, "meta": META_COLS},
    "cleaning": {"duplicates_removed": n_dup, "height_m_to_cm": int(n_height), "weight_jin_to_kg": int(jin.sum()),
                 "sbp_sentinel_to_null": int(n_sentinel), "out_of_range_to_null": n_range},
    "train": {"file": "train.parquet", **describe(train), "content_sha256": content_sha256(train)},
    "valid": {"file": "valid.parquet", **describe(valid), "content_sha256": content_sha256(valid)},
    "train_valid_resident_overlap": len(set(train["resident_id"]) & set(valid["resident_id"])),
    "env": {"python": platform.python_version(), "pandas": pd.__version__},
}
if os.path.exists(FROZEN + "FROZEN.json"):                                # 冻结过：只核对，不覆盖
    old = json.load(open(FROZEN + "FROZEN.json", encoding="utf-8"))
    same = all(old[s]["content_sha256"] == card[s]["content_sha256"] for s in ["train", "valid"])
    assert same, "data/frozen/ 里已经有一份内容不同的冻结集：不要覆盖，先查是哪条口径变了"
    print("已经冻结过，内容一致，不重复写")
else:
    os.makedirs(FROZEN)
    for s in ["train", "valid"]:
        shutil.copyfile(OUT + s + ".parquet", FROZEN + s + ".parquet")     # 冻结的就是验收过的那两份文件
    json.dump(card, open(FROZEN + "FROZEN.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print("已冻结到", FROZEN)
print(sorted(os.listdir(FROZEN)))

已经冻结过，内容一致，不重复写
['FROZEN.json', 'train.parquet', 'valid.parquet']


### 读回来核对：冻结集只认指纹

In [15]:
card = json.load(open(FROZEN + "FROZEN.json", encoding="utf-8"))
for s in ["train", "valid"]:
    d = pd.read_parquet(FROZEN + card[s]["file"])
    assert content_sha256(d) == card[s]["content_sha256"], f"{s}: 读回来的内容和冻结时不一样"
    assert len(d) == card[s]["rows"] and int(d[TARGET].sum()) == card[s]["positives"]
    assert not check_frozen(d), f"{s}: 入口检查没过"
    print(s, card[s]["rows"], card[s]["positives"], card[s]["content_sha256"][:16])
print("来源", card["source"]["file"], card["source"]["sha256"][:16])

train 32632 5285 52e41f28ec7da2b0
valid 13067 1675 b489f7c685ee85ec
来源 incident.db 412b7c1adb612f9f
